# 06 — Integración de voz y compra

Conecta ASR → transcripción revisable → agente/RAG/herramientas → tienda SQLite → respuesta sintetizada con el perfil XTTS preparado en el notebook 02.

El procesamiento se ejecuta localmente en el runtime de Google Colab. No se llama a APIs externas de inferencia.

In [ ]:
%pip -q install -r requirements-lab4-modular.txt

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
from pathlib import Path
from IPython.display import Audio, display
from lab4_asistente import config as C
from lab4_asistente.runtime import cargar_sistema, cargar_modelos_voz


## Cargar el sistema y medir cargas iniciales

In [ ]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]
VOICE_ID = "winz"  # cambia al nombre del WAV y del perfil .pt elegido
ASR, TTS, cargas_voz = cargar_modelos_voz(VOICE_ID, COMP["dispositivo"])
CARGAS_INICIALES = {**COMP["cargas"], **cargas_voz}
print("Cargas iniciales (fuera de la latencia por turno):", CARGAS_INICIALES)


## Procesar una consulta hablada y permitir corregirla

In [ ]:
from google.colab import files

subidos = files.upload()  # selecciona un WAV en español
ruta_audio = Path(next(iter(subidos)))
transcripcion = ASR.transcribir(ruta_audio)
print("Transcripción ASR:", transcripcion)
corregida = input("Corrige la transcripción si hace falta; Enter conserva la original: ").strip()
entrada = corregida or transcripcion

respuesta = SIS.agente.turno(entrada)
print("Respuesta:", respuesta["respuesta"])
print("Operaciones:", [(o["herramienta"], o["resultado"].get("ok")) for o in respuesta["operaciones"]])
ruta_salida = C.RUTA_OUTPUT / "respuesta_demo.wav"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
TTS.sintetizar(respuesta["respuesta"], ruta_salida)
display(Audio(filename=str(ruta_salida)))


## Qué comprobar en la demostración

La transcripción se revisa antes de llegar al agente. El agente solo accede al catálogo sin etiquetas, consulta precios/stock por herramientas y solicita confirmación después de mostrar la cotización. XTTS usa `data/voice_profiles/<VOICE_ID>.pt`; conserva el WAV fuente en `data/voice_samples/<VOICE_ID>.wav` para medir semejanza.